In [1]:
import numpy as np
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
# Configuration
# Loads the cleaned dataset from the embeddings script. 
# The rows are in the same order as the embeddings, so they must not be sorted or filtered again here.
CLEAN_CSV = "posts_clean.csv"
TEXT_COL = "post_text"   
ID_COL = "new_post_id"                            
EMBEDDINGS_PATH ="post_embeddings.npy"
RANDOM_STATE = 1512

In [ ]:
# Loading posts and embeddings

print("Loading posts and embeddings...")
df = pd.read_csv(CLEAN_CSV)
posts = df[TEXT_COL].astype(str).tolist()
embeddings = np.load(EMBEDDINGS_PATH)

# normalize number formatting before posts are used further 
import re

def merge_thousand_separators(text):
    return re.sub(r"(?<=\d)\.(?=\d{3}\b)", "", text)

def merge_hyphenated_numbers(text):
    # "15-årig" -> "15årig", "18-årige" -> "18årige", "3-dobbelt" -> "3dobbelt"
    return re.sub(r"(?<=\d)-(?=[a-zæøåA-ZÆØÅ])", "", text)

posts = [merge_thousand_separators(p) for p in posts]
posts = [merge_hyphenated_numbers(p) for p in posts]
 
assert len(posts) == embeddings.shape[0], (
    f"Mismatch: {len(posts)} posts but {embeddings.shape[0]} embeddings - "
    "Check that the two files originate from the same run and sequence."
)
assert ID_COL in df.columns, (
    f"The column '{ID_COL}' does not exist in {CLEAN_CSV} - "
    "Run the updated BERTopic_embeddings.ipynb script again so that it gets saved."
)
print(f"Loaded {len(posts):,} posts with embeddings of dimension {embeddings.shape[1]}")

In [ ]:
# Danish Stop Words

%pip install stopwordsiso

try:
    import stopwordsiso as stopwords_iso
    danish_stopwords = list(stopwords_iso.stopwords("da"))
except ImportError:
    print("OBS: the package 'stopwordsiso' is not installed "
          "(pip install stopwordsiso) - uses a short NLTK-list instead.")
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    danish_stopwords = stopwords.words("danish")
 
print(f"Uses {len(danish_stopwords)} danish stop words in topic representation.")
 
vectorizer_model = CountVectorizer(
    stop_words=danish_stopwords,
    ngram_range=(1, 2),   # also allows two-word phrases (e.g., "King Frederik") as topic terms
    min_df=5,             # ignores words that appear in fewer than 5 posts (noise reduction)
)

In [ ]:
# UMAP OG HDBSCAN (for reproducibility)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,   # make UMAP deterministic
)
 
hdbscan_model = HDBSCAN(
    min_cluster_size=300,     
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

In [ ]:
# Fit BERTOPIC
# ---------------------------------------------------------------------------
print("\nFitting BERTopic model...")
topic_model = BERTopic(
    embedding_model=None,   # provide own embeddings, so BERTopic does not need to create its own
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="danish",
    calculate_probabilities=False,   
    verbose=True,
)
 
topics, probs = topic_model.fit_transform(posts, embeddings=embeddings)


In [ ]:
# Inspect results

topic_info = topic_model.get_topic_info()
print(f"\nAntal fundne topics (ekskl. outlier-topic -1): "
      f"{len(topic_info[topic_info['Topic'] != -1])}")
 
n_outliers = (np.array(topics) == -1).sum()
pct_outliers = 100 * n_outliers / len(topics)
print(f"Number of posts classified as outliers (topic -1): "
      f"{n_outliers:,} ({pct_outliers:.1f}%)")
 
print("\n- 10 largest topics -")
print(topic_info[topic_info["Topic"] != -1].drop(columns=["Representative_Docs"], errors="ignore").head(10).to_string(index=False))
 
print("\n- Example: words for topic 0 -")
print(topic_model.get_topic(0))

In [ ]:
# Test threshold for outlier reduction

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    test_topics = topic_model.reduce_outliers(
        posts, topics,
        strategy="embeddings",
        embeddings=embeddings,
        threshold=threshold,
    )
    n_outliers = (np.array(test_topics) == -1).sum()
    pct = 100 * n_outliers / len(test_topics)
    print(f"threshold={threshold}: {n_outliers:,} outliers ({pct:.1f}%)")

In [ ]:
new_topics = topic_model.reduce_outliers(
    posts, topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.4,
)

topic_model.update_topics(posts, topics=new_topics, vectorizer_model=vectorizer_model)

n_outliers_new = (np.array(new_topics) == -1).sum()
print(f"Outliers after reduction: {n_outliers_new:,} ({100*n_outliers_new/len(new_topics):.1f}%)")

In [ ]:
print(topic_model.get_topic_info().drop(columns=["Representative_Docs"], errors="ignore").sort_values("Count", ascending=False).to_string(index=False))

In [ ]:
# Re-numbering topics after outlier reduction step

from collections import Counter

# Renumber: 0 = largest topic, -1 remains an outlier
counts = Counter(t for t in topic_model.topics_ if t != -1)
old_to_new = {old: new for new, (old, _) in enumerate(counts.most_common())}
old_to_new[-1] = -1

topic_model.update_topics(
    posts,
    topics=[old_to_new[t] for t in topic_model.topics_],
    vectorizer_model=vectorizer_model,
)

print(topic_model.get_topic_info().drop(columns=["Representative_Docs"], errors="ignore").sort_values("Topic").to_string(index=False))

In [ ]:
# Writing out results

topic_info = topic_model.get_topic_info()
topic_info.to_excel("bertopic_resultater.xlsx", index=False)

In [ ]:
# Writing out topic id's to merge onto dataset

df_export = pd.DataFrame({
    "new_post_id": df[ID_COL],
    "topic_id": topic_model.topics_,
})

df_export.to_csv("posts_topic_ids.csv", index=False)

In [ ]:
# Inspecting topic 0

topic_0_docs = topic_model.get_representative_docs(0)

for i, doc in enumerate(topic_0_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc)

In [ ]:
# Inspecting topic 0

topic_0_docs = topic_model.get_document_info(df["post_text"].tolist())

topic_0_docs = topic_0_docs[
    topic_0_docs["Topic"] == 0
].sort_values("Probability", ascending=False)

print(topic_0_docs[["Document", "Probability"]].head(30).to_string(index=False))